### 下載imdb資料

In [8]:
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz
!rm -r aclImdb/train/unsup

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 80.2M  100 80.2M    0     0   308k      0  0:04:26  0:04:26 --:--:--  874k     0   120k      0  0:11:19  0:00:07  0:11:12  165k0.2M   21 17.1M    0     0   171k      0  0:07:57  0:01:42  0:06:15  137k     0   156k      0  0:08:46  0:02:16  0:06:30  230k    0   153k      0  0:08:54  0:02:25  0:06:29  124k  495k  268k      0  0:05:06  0:04:07  0:00:59  720k


In [9]:
import os, pathlib, shutil, random

base_dir = pathlib.Path("aclImdb")
val_dir = base_dir / "val"
train_dir = base_dir / "train"
for category in ("neg", "pos"):
    os.makedirs(val_dir / category)
    files = os.listdir(train_dir / category)
    random.Random(1337).shuffle(files)
    num_val_samples = int(0.2 * len(files))
    val_files = files[-num_val_samples:]
    for fname in val_files:
        shutil.move(train_dir / category / fname,
                    val_dir / category / fname)

FileExistsError: [Errno 17] File exists: 'aclImdb/val/neg'

In [ ]:
from tensorflow import keras
batch_size = 32

train_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/train", batch_size=batch_size
)
val_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/val", batch_size=batch_size
)
test_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/test", batch_size=batch_size
)

text_only_train_ds = train_ds.map(lambda x, y: x)

### 文字預處理

In [ ]:
from tensorflow.keras import layers

max_length = 600
max_tokens = 20000
text_vectorization = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=max_length,
)
text_vectorization.adapt(text_only_train_ds)

int_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
int_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
int_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

## 以下區塊請同學自由發揮

In [ ]:



# Embeddingのinput_dimとoutput_dimを指定
embedding_dim = 128
inputs = keras.Input(shape=(None,), dtype="int64")
embedded = layers.Embedding(input_dim=max_tokens, output_dim=embedding_dim)(inputs)
# ...（残りのモデルの構築）

outputs = layers.Dense(1, activation='sigmoid')(x)  # 2クラス分類なのでsigmoidを使用

model = keras.Model(inputs, outputs)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=["accuracy"])
model.summary()

### 訓練與測試準確度

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint("hw_2",
                    save_best_only=True)
]
model.fit(int_train_ds, validation_data=int_val_ds, epochs=10, callbacks=callbacks)
model = keras.models.load_model("hw_2")
print(f"Test acc: {model.evaluate(int_test_ds)[1]:.3f}")

### 隱藏層0個
隱藏層的數量是在代碼中，未直接指定隱藏層的數量，僅使用了嵌入層（Embedding）。因此，該模型似乎由單一的嵌入層和密集層（Dense）組成。未提供隱藏層的具體數量或結構。


### 優化器
在提供的程式碼中，使用的優化器是Adam。
在這裡，optimizer 參數被設置為 'adam'，這意味著使用Adam優化器。Adam是一種常用的優化算法，特別適用於訓練神經網絡。它結合了動量（momentum）和適應性學習率（adaptive learning rate）的概念，以提高模型的收斂速度並更有效地適應不同特徵的梯度。
簡而言之，這段程式碼使用Adam作為模型的優化器，用於最小化訓練過程中的二元交叉熵損失（binary_crossentropy），同時監視模型的準確性（accuracy）。

### 損失函數
損失函數是模型編譯中使用的損失函數是binary_crossentropy。這是適用於2類別二元分類問題的損失函數，它試圖最小化模型輸出的概率分佈與真實分佈（0或1）之間的交叉熵。


### 斷詞方法
斷詞方法是代碼中未提供斷詞方法。然而，在將文本數據提供給模型之前，使用了TextVectorization層。該層用於將文本轉換為整數序列，可以視為斷詞（tokenization）的一部分。詳細的tokenization方法和參數在代碼中指定。
